In [ ]:
import numpy as np
import pandas as pd
import math
import json
from tqdm.notebook import tqdm
import warnings

# --- Stacking Model Imports ---
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFECV
warnings.filterwarnings('ignore')
from catboost import CatBoostClassifier

# --- Data File Paths ---
train_file_path = '../input/fds-pokemon-battles-prediction-2025/train.jsonl'
test_file_path = '../input/fds-pokemon-battles-prediction-2025/test.jsonl'

I loaded the data beforehand to process a little bit.

In [ ]:
import json
import pandas as pd
import os

# --- Define the path to our data ---
COMPETITION_NAME = "fds-pokemon-battles-prediction-2025"
DATA_PATH = os.path.join("../input", COMPETITION_NAME)

train_file_path = os.path.join(DATA_PATH, "train.jsonl")
test_file_path = os.path.join(DATA_PATH, "test.jsonl")
train_data_inspect = []

# Read the file line by line
print(f"Loading data from '{train_file_path}'...")
try:
    with open(train_file_path, "r") as f:
        for line in f:
            # json.loads() parses one line (one JSON object) into a Python dictionary
            train_data_inspect.append(json.loads(line))

    print(f"Successfully loaded {len(train_data_inspect)} battles.")

    # Inspect the first battle to see its structure
    print("\n--- Structure of the first train battle: ---")
    if train_data_inspect:
        first_battle = train_data_inspect[1]

        # To keep the output clean, we can create a copy and truncate the timeline
        battle_for_display = first_battle.copy()
        battle_for_display["battle_timeline"] = battle_for_display.get(
            "battle_timeline", []
        )[
            :30
        ]  # Show first 30 turns

        # Use json.dumps for pretty-printing the dictionary
        print(json.dumps(battle_for_display, indent=4))
        if len(first_battle.get("battle_timeline", [])) > 3:
            print("    ...")
            print("    (battle_timeline has been truncated for display)")


except FileNotFoundError:
    print(f"ERROR: Could not find the training file at '{train_file_path}'.")
    print("Please make sure you have added the competition data to this notebook.")


In [ ]:
# --- Load Raw Data ---
train_data = []
print(f"Loading data from '{train_file_path}'...")
try:
    with open(train_file_path, "r") as f:
        for line in f:
            train_data.append(json.loads(line))
    print(f"Successfully loaded {len(train_data)} train battles.")

    test_data = []
    with open(test_file_path, "r") as f:
        for line in f:
            test_data.append(json.loads(line))
    print(f"Successfully loaded {len(test_data)} test battles.")

except FileNotFoundError:
    print(
        f"ERROR: Could not find data files at '{train_file_path}' or '{test_file_path}'."
    )


In [ ]:
def my_func(data):
    pokemon_db = {}
    play_win_stats = {}
    total_wins = 0
    for battle in tqdm(data, desc="A little preprocessing"):
        h = set(
            [
                t.get("p2_pokemon_state", {}).get("name", "unknown")
                for t in battle.get("battle_timeline")
            ]
        )
        for p in battle.get("p1_team_details", []):
            name = p.get("name")
            if name:
                pokemon_db[name.lower()] = {
                    "base_hp": p.get("base_hp", 0),
                    "base_atk": p.get("base_atk", 0),
                    "base_def": p.get("base_def", 0),
                    "base_spa": p.get("base_spa", 0),
                    "base_spd": p.get("base_spd", 0),
                    "base_spe": p.get("base_spe", 0),
                    "types": p.get("types", []),
                }
            if name not in h:
                if name in play_win_stats.keys():
                    play_win_stats[name]["played"] += 1
                    if battle.get("player_won") == True:
                        play_win_stats[name]["num_of_win"] += 1
                        total_wins += 1
                else:
                    play_win_stats[name] = {}
                    play_win_stats[name]["played"] = 1
                    if battle.get("player_won") == True:
                        play_win_stats[name]["num_of_win"] = 1
                        total_wins += 1
                    else:
                        play_win_stats[name]["num_of_win"] = 0
    for key in play_win_stats:
        play_win_stats[key]["ratio"] = (
            play_win_stats[key]["num_of_win"] / play_win_stats[key]["played"]
        )
    play_win_stats = dict(
        sorted(play_win_stats.items(), key=lambda item: item[1]["ratio"])
    )
    loser_pokemons = [
        name for name in play_win_stats if play_win_stats[name]["ratio"] < 0.4
    ]
    return pokemon_db, loser_pokemons


pk_vals, losers = my_func(train_data)


In [ ]:
def create_golden_features_V18(data: list[dict]):
    """
    V18 Feature Set.
    1. Base: V3 "Elite" feature set.
    2. Golden Feature: 'team_hp_total_diff'.
    3. Golden Feature: 'pokemon_seen_diff'.
    4. Golden Feature: 'fnt_count_diff'.
    """

    # ---- Gen I Type Chart ----
    TYPE_CHART = {
        "normal": {"rock": 0.5, "ghost": 0.0},
        "fire": {
            "fire": 0.5,
            "water": 0.5,
            "grass": 2,
            "ice": 2,
            "bug": 2,
            "rock": 0.5,
            "dragon": 0.5,
        },
        "water": {
            "fire": 2,
            "water": 0.5,
            "grass": 0.5,
            "ground": 2,
            "rock": 2,
            "dragon": 0.5,
        },
        "electric": {
            "water": 2,
            "electric": 0.5,
            "grass": 0.5,
            "ground": 0,
            "flying": 2,
            "dragon": 0.5,
        },
        "grass": {
            "fire": 0.5,
            "water": 2,
            "grass": 0.5,
            "poison": 0.5,
            "ground": 2,
            "flying": 0.5,
            "bug": 0.5,
            "rock": 2,
            "dragon": 0.5,
        },
        "ice": {
            "water": 0.5,
            "grass": 2,
            "ice": 0.5,
            "ground": 2,
            "flying": 2,
            "dragon": 2,
        },
        "fighting": {
            "normal": 2,
            "ice": 2,
            "poison": 0.5,
            "flying": 0.5,
            "psychic": 0.5,
            "bug": 0.5,
            "rock": 2,
            "ghost": 0,
        },
        "poison": {
            "grass": 2,
            "poison": 0.5,
            "ground": 0.5,
            "bug": 2,
            "rock": 0.5,
            "ghost": 0.5,
        },
        "ground": {
            "fire": 2,
            "electric": 2,
            "grass": 0.5,
            "poison": 2,
            "flying": 0,
            "bug": 0.5,
            "rock": 2,
        },
        "flying": {"electric": 0.5, "grass": 2, "fighting": 2, "bug": 2, "rock": 0.5},
        "psychic": {"fighting": 2, "poison": 2, "psychic": 0.5, "ghost": 0.0},
        "bug": {
            "fire": 0.5,
            "grass": 2,
            "fighting": 0.5,
            "poison": 2,
            "flying": 0.5,
            "psychic": 0.5,
            "ghost": 0.5,
        },
        "rock": {
            "fire": 2,
            "ice": 2,
            "fighting": 0.5,
            "ground": 0.5,
            "flying": 2,
            "bug": 2,
        },
        "ghost": {"normal": 0, "psychic": 0.0, "ghost": 2},
        "dragon": {"dragon": 2},
        "notype": {},
    }
    TYPE_CHART.setdefault("bug", {}).update({"poison": 2.0})
    TYPE_CHART.setdefault("poison", {}).update({"bug": 2.0})
    KNOWN_TYPES = set(TYPE_CHART.keys())

    # ---- Move Categories ----
    STATUS_MOVES = {
        "sleeppowder",
        "sing",
        "stunspore",
        "thunderwave",
        "toxic",
        "glare",
        "poisonpowder",
        "spore",
        "supersonic",
    }
    BOOST_MOVES = {
        "amnesia",
        "swordsdance",
        "agility",
        "barrier",
        "doubleteam",
        "growth",
        "harden",
        "meditate",
        "minimize",
    }
    FIXED_DAMAGE_MOVES = {"seismictoss", "nightshade", "sonicboom", "psywave"}
    OHKO_MOVES = {"fissure", "horndrill", "guillotine"}
    DIRECT_HEALING_MOVES = {"recover", "softboiled"}
    REST_MOVE = {"rest"}
    DRAIN_MOVES = {"megadrain"}
    T1_STATUS_MOVES = {"sleeppowder", "spore", "thunderwave", "toxic"}
    T1_RECOVERY_MOVES = {"recover", "softboiled", "rest"}

    # ---- Helper Functions ----
    def _clean_types(ts):
        return [
            (t or "").lower() if (t or "").lower() in KNOWN_TYPES else "notype"
            for t in (ts or [])
        ]

    def _type_mult(attacking: str, defending_types: list[str]) -> float:
        attacking = (attacking or "").lower()
        out = 1.0
        if not defending_types:
            return 1.0
        table = TYPE_CHART.get(attacking, {})
        for d in defending_types:
            out *= table.get((d or "").lower(), 1.0)
        return out

    def _stab(move_type: str, attacker_types: list[str]) -> float:
        m = (move_type or "").lower()
        return 1.5 if m and any(t.lower() == m for t in (attacker_types or [])) else 1.0

    def _exp_damage(
        move: dict, atk_types: list[str], def_types: list[str], cap: float = 200.0
    ) -> float:
        if not move:
            return 0.0
        name = (move.get("name") or "").lower()
        base = float(move.get("base_power", 0) or 0.0)
        acc = float(move.get("accuracy", 1.0) or 1.0)
        mtype = (move.get("type") or "").lower()
        if name in FIXED_DAMAGE_MOVES:
            return min(100.0 * acc, cap)
        if name in OHKO_MOVES:
            return min(250.0 * acc, cap)
        val = base * acc * _type_mult(mtype, def_types) * _stab(mtype, atk_types)
        return min(val, cap)

    # ---- Main Feature Loop ----
    rows = []
    ctr = 0
    for b in tqdm(data, desc="Extracting V18 (Golden) features"):

        # Get P2 team from first 30 turns
        p2_team = {}
        p1_played_team = {}
        if b.get("battle_timeline"):
            for turn in b.get("battle_timeline"):
                if turn.get("p2_pokemon_state"):
                    d_name = turn.get("p2_pokemon_state").get("name", "unknown")
                    if d_name not in p2_team.keys():
                        p2_team[d_name] = {}
                        p2_team[d_name]["base_hp"] = pk_vals[d_name].get("base_hp", 0)
                        p2_team[d_name]["base_atk"] = pk_vals[d_name].get("base_atk", 0)
                        p2_team[d_name]["base_def"] = pk_vals[d_name].get("base_def", 0)
                        p2_team[d_name]["base_spa"] = pk_vals[d_name].get("base_spa", 0)
                        p2_team[d_name]["base_spe"] = pk_vals[d_name].get("base_spe", 0)
                        p2_team[d_name]["types"] = pk_vals[d_name].get(
                            "types", "unknown"
                        )

                if turn.get("p1_pokemon_state"):
                    d_name2 = turn.get("p1_pokemon_state").get("name", "unknown")
                    if d_name2 not in p1_played_team.keys():
                        p1_played_team[d_name2] = {}
                        p1_played_team[d_name2]["base_hp"] = pk_vals[d_name2].get(
                            "base_hp", 0
                        )
                        p1_played_team[d_name2]["base_atk"] = pk_vals[d_name2].get(
                            "base_atk", 0
                        )
                        p1_played_team[d_name2]["base_def"] = pk_vals[d_name2].get(
                            "base_def", 0
                        )
                        p1_played_team[d_name2]["base_spa"] = pk_vals[d_name2].get(
                            "base_spa", 0
                        )
                        p1_played_team[d_name2]["base_spe"] = pk_vals[d_name2].get(
                            "base_spe", 0
                        )
                        p1_played_team[d_name2]["types"] = pk_vals[d_name2].get(
                            "types", "unknown"
                        )

        p1_num_of_losers = 0
        p2_num_of_losers = 0
        for name in p1_played_team:
            if name in losers:
                p1_num_of_losers += 1
        for name in p2_team:
            if name in losers:
                p2_num_of_losers += 1
        loser_diff = p1_num_of_losers - p2_num_of_losers

        r = {}
        p1_team = b.get("p1_team_details") or []
        p2_lead = b.get("p2_lead_details") or {}
        tl = b.get("battle_timeline") or []
        n_turns = len(tl) or 1
        # --- Static Features ---
        if p1_team:
            r["p1_mean_hp"] = float(np.mean([p.get("base_hp", 0) for p in p1_team]))
            r["p1_mean_atk"] = float(np.mean([p.get("base_atk", 0) for p in p1_team]))
            r["p1_mean_def"] = float(np.mean([p.get("base_def", 0) for p in p1_team]))
            r["p1_mean_spa"] = float(np.mean([p.get("base_spa", 0) for p in p1_team]))
            r["p1_mean_spe"] = float(np.mean([p.get("base_spe", 0) for p in p1_team]))
        else:
            r["p1_mean_hp"] = 0.0
            r["p1_mean_atk"] = 0.0
            r["p1_mean_def"] = 0.0
            r["p1_mean_spa"] = 0.0
            r["p1_mean_spe"] = 0.0
        if p2_team:
            r["p2_mean_hp"] = float(
                np.mean([p2_team[p].get("base_hp", 0) for p in p2_team])
            )
            r["p2_mean_atk"] = float(
                np.mean([p2_team[p].get("base_atk", 0) for p in p2_team])
            )
            r["p2_mean_def"] = float(
                np.mean([p2_team[p].get("base_def", 0) for p in p2_team])
            )
            r["p2_mean_spa"] = float(
                np.mean([p2_team[p].get("base_spa", 0) for p in p2_team])
            )
            r["p2_mean_spe"] = float(
                np.mean([p2_team[p].get("base_spe", 0) for p in p2_team])
            )
        else:
            r["p2_mean_hp"] = 0.0
            r["p2_mean_atk"] = 0.0
            r["p2_mean_def"] = 0.0
            r["p2_mean_spa"] = 0.0
            r["p2_mean_spe"] = 0.0

        r["p2_lead_hp"] = p2_lead.get("base_hp", 0) or 0
        r["p2_lead_atk"] = p2_lead.get("base_atk", 0) or 0
        r["p2_lead_def"] = p2_lead.get("base_def", 0) or 0
        r["p2_lead_spe"] = p2_lead.get("base_spe", 0) or 0

        p1_lead_hp = (b.get("p1_lead_details") or {}).get("base_hp", 0) or 0
        p1_lead_atk = (b.get("p1_lead_details") or {}).get("base_atk", 0) or 0
        p1_lead_spe = (b.get("p1_lead_details") or {}).get("base_spe", 0) or 0

        r["lead_hp_diff"] = p1_lead_hp - r["p2_lead_hp"]
        r["lead_atk_diff"] = p1_lead_atk - r["p2_lead_atk"]
        r["lead_speed_diff"] = p1_lead_spe - r["p2_lead_spe"]

        r["mean_hp_diff"] = float(
            np.mean([p.get("base_hp", 0) for p in p1_team])
        ) - float(np.mean([p2_team[p].get("base_hp", 0) for p in p2_team]))
        r["mean_atk_diff"] = float(
            np.mean([p.get("base_atk", 0) for p in p1_team])
        ) - float(np.mean([p2_team[p].get("base_atk", 0) for p in p2_team]))
        r["mean_speed_diff"] = float(
            np.mean([p.get("base_spe", 0) for p in p1_team])
        ) - float(np.mean([p2_team[p].get("base_spe", 0) for p in p2_team]))

        r["loser_diff"] = loser_diff

        # --- Dynamic Features Init ---
        mid_hp_diff = 0.0
        final_boost_advantage = 0.0
        p1_exp, p2_exp = 0.0, 0.0
        p1_sw, p2_sw = 0, 0
        p1_remaining_pokemons = {}

        # Use a dictionary to track the status of each Pokémon
        p1_team_status = {}  # e.g.: {'pikachu': {'hp': 0.5, 'fainted': False}}
        p2_team_status = {}  # e.g.: {'snorlax': {'hp': 0.1, 'fainted': False}}

        p1_stat, p2_stat = 0, 0
        p1_boost, p2_boost = 0, 0
        p1_dmg, p2_dmg = 0, 0
        p1_lost, p2_lost = 0.0, 0.0
        p1_slp = p2_slp = 0
        p1_frz = p2_frz = 0
        p1_psn = p2_psn = 0
        p1_par = p2_par = 0
        p1_brn = p2_brn = 0
        p1_dheal = p2_dheal = 0.0
        p1_drain = p2_drain = 0.0
        p1_t1_status, p2_t1_status = 0, 0
        p1_t1_recover, p2_t1_recover = 0, 0

        if len(tl) > 0:
            mid_idx = len(tl) // 2
            mid_hp_diff = tl[mid_idx].get("p1_pokemon_state", {}).get(
                "hp_pct", 0.0
            ) - tl[mid_idx].get("p2_pokemon_state", {}).get("hp_pct", 0.0)
            for i, turn in enumerate(tl):
                ps1 = turn.get("p1_pokemon_state") or {}
                ps2 = turn.get("p2_pokemon_state") or {}
                s1 = ps1.get("status", "nostatus")
                s2 = ps2.get("status", "nostatus")

                # --- V18 Logic: Update team status ---
                if ps1 and ps1.get("name"):
                    hp = ps1.get("hp_pct", 0.0)
                    p1_team_status[ps1.get("name")] = {"hp": hp, "fainted": (hp <= 0.0)}
                    p1_remaining_pokemons[ps1.get("name")] = {}
                    p1_remaining_pokemons[ps1.get("name")]["hp_pct"] = hp
                if ps2 and ps2.get("name"):
                    hp = ps2.get("hp_pct", 0.0)
                    p2_team_status[ps2.get("name")] = {"hp": hp, "fainted": (hp <= 0.0)}

                if s1 == "slp":
                    p1_slp += 1
                elif s1 == "frz":
                    p1_frz += 1
                elif s1 in ("psn", "tox"):
                    p1_psn += 1
                elif s1 == "par":
                    p1_par += 1
                elif s1 == "brn":
                    p1_brn += 1
                if s2 == "slp":
                    p2_slp += 1
                elif s2 == "frz":
                    p2_frz += 1
                elif s2 in ("psn", "tox"):
                    p2_psn += 1
                elif s2 == "par":
                    p2_par += 1
                elif s2 == "brn":
                    p2_brn += 1
                if i > 0:
                    prev1 = tl[i - 1].get("p1_pokemon_state") or {}
                    prev2 = tl[i - 1].get("p2_pokemon_state") or {}
                    if ps1.get("name") == prev1.get("name"):
                        h1, h1p = (
                            ps1.get("hp_pct", 1.0) or 0.0,
                            prev1.get("hp_pct", 1.0) or 0.0,
                        )
                        p1_lost += (h1p - h1) if h1 < h1p else 0
                    if ps2.get("name") == prev2.get("name"):
                        h2, h2p = (
                            ps2.get("hp_pct", 1.0) or 0.0,
                            prev2.get("hp_pct", 1.0) or 0.0,
                        )
                        p2_lost += (h2p - h2) if h2 < h2p else 0
                if turn.get("p1_move_details") is None:
                    p1_sw += 1
                if turn.get("p2_move_details") is None:
                    p2_sw += 1
                m1 = turn.get("p1_move_details")
                m2 = turn.get("p2_move_details")
                if m1 and ps2:
                    atk = _clean_types(ps1.get("types", []))
                    dft = _clean_types(ps2.get("types", []))
                    p1_exp += _exp_damage(m1, atk, dft)
                    n1 = (m1.get("name") or "").lower()
                    if n1 in DIRECT_HEALING_MOVES:
                        p1_dheal += 0.5
                    elif n1 in DRAIN_MOVES:
                        p1_drain += 0.1
                    if n1 in STATUS_MOVES:
                        p1_stat += 1
                    elif n1 in BOOST_MOVES:
                        p1_boost += 1
                    elif (m1.get("base_power", 0) or 0) > 0:
                        p1_dmg += 1
                    if n1 in T1_STATUS_MOVES:
                        p1_t1_status += 1
                    if n1 in T1_RECOVERY_MOVES:
                        p1_t1_recover += 1
                if m2 and ps1:
                    atk = _clean_types(ps2.get("types", []))
                    dft = _clean_types(ps1.get("types", []))
                    p2_exp += _exp_damage(m2, atk, dft)
                    n2 = (m2.get("name") or "").lower()
                    if n2 in DIRECT_HEALING_MOVES:
                        p2_dheal += 0.5
                    elif n2 in DRAIN_MOVES:
                        p2_drain += 0.1
                    if n2 in STATUS_MOVES:
                        p2_stat += 1
                    elif n2 in BOOST_MOVES:
                        p2_boost += 1
                    elif (m2.get("base_power", 0) or 0) > 0:
                        p2_dmg += 1
                    if n2 in T1_STATUS_MOVES:
                        p2_t1_status += 1
                    if n2 in T1_RECOVERY_MOVES:
                        p2_t1_recover += 1

            final_boost_advantage = sum((ps1.get("boosts", {}) or {}).values()) - sum(
                (ps2.get("boosts", {}) or {}).values()
            )

        p1_all_names = [p.get("name", "unknown") for p in p1_team]
        for name in p1_all_names:
            if name not in p1_remaining_pokemons.keys():
                p1_remaining_pokemons[name] = {}
                p1_remaining_pokemons[name]["hp_pct"] = 1.0

        p1_remaining_pokemons = {
            name: val
            for name, val in p1_remaining_pokemons.items()
            if val.get("hp_pct", 0) > 0
        }

        for k, v in p1_remaining_pokemons.items():
            p1_remaining_pokemons[k]["rem_hp"] = p1_remaining_pokemons[k][
                "hp_pct"
            ] * pk_vals[k].get("base_hp", 0)
            p1_remaining_pokemons[k]["base_atk"] = pk_vals[k].get("base_atk", 0)
            p1_remaining_pokemons[k]["base_def"] = pk_vals[k].get("base_def", 0)
            p1_remaining_pokemons[k]["base_spa"] = pk_vals[k].get("base_spa", 0)
            p1_remaining_pokemons[k]["base_spe"] = pk_vals[k].get("base_spe", 0)
            p1_remaining_pokemons[k]["types"] = pk_vals[k].get("types", 0)

        r["remaining_total_hp"] = np.sum(
            [
                p1_remaining_pokemons[p].get("rem_hp")
                for p in p1_remaining_pokemons.keys()
            ]
        )
        r["remaining_poke_speed"] = np.sum(
            [
                p1_remaining_pokemons[p].get("base_spe")
                for p in p1_remaining_pokemons.keys()
            ]
        )

        rem_type_list = list(
            {
                tp
                for info in p1_remaining_pokemons.values()
                for tp in info.get("types", [])
            }
        )
        if "notype" in rem_type_list:
            rem_type_list.remove("notype")

        # --- Golden Feature V18: Team Health & Faint Count ---

        # 1. Faint Count
        p1_fnt_count = sum(1 for p in p1_team_status.values() if p["fainted"])
        p2_fnt_count = sum(1 for p in p2_team_status.values() if p["fainted"])
        r["fnt_count_diff"] = p1_fnt_count - p2_fnt_count

        # 2. Total Health
        # Sum HP% only for non-fainted Pokémon
        p1_total_hp = sum(p["hp"] for p in p1_team_status.values() if not p["fainted"])
        p2_total_hp = sum(p["hp"] for p in p2_team_status.values() if not p["fainted"])

        # Calculate "unseen" Pokémon (not fainted and not seen)
        p1_unseen_count = 6 - len(p1_team_status)
        p2_unseen_count = 6 - len(p2_team_status)

        r["team_hp_total_p1"] = p1_total_hp + p1_unseen_count
        r["team_hp_total_p2"] = p2_total_hp + p2_unseen_count
        r["team_hp_total_diff"] = r["team_hp_total_p1"] - r["team_hp_total_p2"]

        # --- Aggregate Timeline Features ---
        r["mid_hp_diff"] = float(mid_hp_diff)
        r["final_boost_advantage"] = float(final_boost_advantage)
        r["expected_power_diff"] = float(max(min(p1_exp - p2_exp, 2000.0), -2000.0))
        r["switch_diff"] = float(p1_sw - p2_sw) / n_turns

        # 3. Pokémon Seen Count
        r["pokemon_seen_diff"] = (
            float(len(p1_team_status)) - float(len(p2_team_status)) / n_turns
        )

        r["status_moves_diff"] = float(p1_stat - p2_stat) / n_turns
        r["boost_moves_diff"] = float(p1_boost - p2_boost) / n_turns
        r["damage_moves_diff"] = float(p1_dmg - p2_dmg) / n_turns
        r["total_hp_lost_diff"] = float(p2_lost - p1_lost)
        r["slp_frz_turns_adv"] = float((p2_slp + p2_frz) - (p1_slp + p1_frz)) / n_turns
        r["psn_tox_turns_adv"] = float(p2_psn - p1_psn) / n_turns
        r["lost_speed_turns_adv"] = float(p2_par - p1_par) / n_turns
        r["reduced_atk_turns_adv"] = float(p2_brn - p1_brn) / n_turns
        r["healing_adv"] = (
            float((p1_dheal + p1_drain) - (p2_dheal + p2_drain)) / n_turns
        )

        # --- V2 "Golden" Features ---
        r["t1_status_adv"] = float(p1_t1_status - p2_t1_status) / n_turns
        r["t1_recover_adv"] = float(p1_t1_recover - p2_t1_recover) / n_turns

        r["battle_id"] = b.get("battle_id")
        r["player_won"] = int(b["player_won"]) if "player_won" in b else 0
        rows.append(r)

    df = pd.DataFrame(rows).fillna(0)
    for c in df.select_dtypes(include=["float"]).columns:
        df[c] = df[c].astype("float32")
    for c in df.select_dtypes(include=["int"]).columns:
        if c != "player_won":
            df[c] = df[c].astype("int32")
    return df


In [ ]:
# --- Process Features ---
if "train_data" in locals() and "test_data" in locals():
    print("\nProcessing training data...")
    # --- USE V18 FUNCTION ---
    train_df = create_golden_features_V18(train_data)

    print("\nProcessing test data...")
    # --- USE V18 FUNCTION ---
    test_df = create_golden_features_V18(test_data)

    # Drop the target from test_df
    if "player_won" in test_df.columns:
        test_df = test_df.drop(columns=["player_won"])

    print(f"\nCreated {len(train_df.columns) - 2} features.")
else:
    print("\nSkipping feature processing due to missing data.")


In [ ]:
if "train_df" in locals():
    # --- Define Level 1 Base Models ---
    # Use un-tuned models for better stacking performance
    model_xgb = Pipeline(
        [
            (
                "model",
                xgb.XGBClassifier(
                    colsample_bytree=0.8,
                    learning_rate=0.1,
                    max_depth=3,
                    n_estimators=250,
                    subsample=0.7,
                    random_state=42,
                    eval_metric="logloss",
                    use_label_encoder=False,
                    n_jobs=-1,
                ),
            )
        ]
    )
    model_gb = Pipeline(
        [
            (
                "model",
                GradientBoostingClassifier(
                    n_estimators=250, random_state=42, max_depth=3, learning_rate=0.1
                ),
            )
        ]
    )
    model_lr = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)),
        ]
    )

    model_cb = Pipeline(
        [
            (
                "model",
                CatBoostClassifier(
                    iterations=1000,
                    random_state=42,
                    max_depth=4,
                    learning_rate=0.1,
                    verbose=0,
                ),
            )
        ]
    )
    base_models = {"xgb": model_xgb, "gb": model_gb, "lr": model_lr, "cb": model_cb}

    print("\n--- Starting Stacking (on V18 Features) ---")

    # --- Define features and target ---
    target = "player_won"
    features_v18 = [col for col in train_df.columns if col not in ["battle_id", target]]
    X_train_v18 = train_df[features_v18]
    y_train = train_df[target]
    X_test_v18 = test_df[features_v18]

    # Align columns just in case
    X_test_v18 = X_test_v18[X_train_v18.columns]

    meta_features_train = pd.DataFrame()
    meta_features_test = pd.DataFrame()
    N_SPLITS_STACK = 5
    skf_stack = StratifiedKFold(n_splits=N_SPLITS_STACK, shuffle=True, random_state=42)

    for model_name, model in base_models.items():
        print(f"Training {model_name} on V18 features...")

        # 1. Get OOF predictions for the training data
        oof_preds = cross_val_predict(
            model, X_train_v18, y_train, cv=skf_stack, method="predict_proba", n_jobs=-1
        )[:, 1]
        meta_features_train[f"pred_{model_name}"] = oof_preds

        # 2. Train on ALL training data and predict on test set
        model.fit(X_train_v18, y_train)
        test_preds = model.predict_proba(X_test_v18)[:, 1]
        meta_features_test[f"pred_{model_name}"] = test_preds

    print("\n--- Level 1 OOF Predictions Generated ---")

    # --- Train the Level 2 Meta-Model ---
    print("\n--- Training Level 2 Meta-Model ---")
    meta_model = LogisticRegression(random_state=42, n_jobs=-1)

    # Fit the final meta-model on all OOF predictions
    meta_model.fit(meta_features_train, y_train)


In [ ]:
if "meta_model" in locals():
    print("\n--- Finding optimal threshold for the STACKED model ---")

    # Get OOF predictions for the meta-model
    oof_probabilities_stacked = cross_val_predict(
        meta_model,
        meta_features_train,  # Train on the meta-features
        y_train,
        cv=skf_stack,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    print("Searching for the best accuracy threshold...")
    thresholds = np.linspace(0.3, 0.7, 81)
    accuracies = []
    for thr in thresholds:
        oof_preds_at_thr = (oof_probabilities_stacked >= thr).astype(int)
        acc = accuracy_score(y_train, oof_preds_at_thr)
        accuracies.append(acc)

    best_thr_idx = np.argmax(accuracies)
    optimal_threshold_stacked = thresholds[best_thr_idx]
    best_accuracy_stacked = accuracies[best_thr_idx]

    print(f"\nOptimal stacked threshold found: {optimal_threshold_stacked:.4f}")
    print(f"Stacked accuracy at optimal threshold: {best_accuracy_stacked:.5f}")


In [ ]:
if "meta_model" in locals():
    print(f"Models are already trained.")
    print("Predicting probabilities on the test set meta-features...")

    # Predict on the test set's meta-features
    test_probabilities_stacked = meta_model.predict_proba(meta_features_test)[:, 1]

    # Apply the OPTIMAL Stacked Threshold
    print(f"Applying optimal stacked threshold: {optimal_threshold_stacked:.4f}")
    final_predictions = (
        test_probabilities_stacked >= optimal_threshold_stacked
    ).astype(int)

    # Create the Final Submission File
    print("\nCreating final submission file...")
    submission_df = pd.DataFrame(
        {"battle_id": test_df["battle_id"], "player_won": final_predictions}
    )
    submission_df.to_csv("submission.csv", index=False)

    print("\n'submission.csv' file created successfully!")
    display(submission_df.head())
